## Imports


In [104]:
import os
import json
from datetime import datetime
from langdetect import detect, LangDetectException
from sentence_transformers import SentenceTransformer
import faiss
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score

## Configs

### Datasets

In [105]:
DIRECTORY = Path().resolve().parent

In [106]:
MAPPER_PATH = DIRECTORY / "Datasets" / "mapper" / "mapper.json"
TESTER_PATH = DIRECTORY / "Datasets" / "mapper" / "test.json"
MAPPER_PATH, TESTER_PATH

(WindowsPath('E:/Studies/MIT/7/FYP/Project/Datasets/mapper/mapper.json'),
 WindowsPath('E:/Studies/MIT/7/FYP/Project/Datasets/mapper/test.json'))

### Parameters

In [120]:
score_threshold = 0.60

### Model Name

In [108]:
# model_name = "paraphrase-mpnet-base-v2"
model_name = "all-MiniLM-L6-v2"

### Saving Results

In [109]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

In [110]:
SAVE_PATH = DIRECTORY / "Results" / "CaneSpeak" / model_name / timestamp
save_path_testing = os.path.join(SAVE_PATH, f"testing.json")   
save_path_metrics = os.path.join(SAVE_PATH, f"metrics.txt")
print(f"The testing results will be saved to: {save_path_testing}")
print(f"The result metrics will be saved to: {save_path_metrics}")

The testing results will be saved to: E:\Studies\MIT\7\FYP\Project\Results\CaneSpeak\all-MiniLM-L6-v2\2025-11-18_19-57\testing.json
The result metrics will be saved to: E:\Studies\MIT\7\FYP\Project\Results\CaneSpeak\all-MiniLM-L6-v2\2025-11-18_19-57\metrics.txt


## Loading Datasets and Model

### Query-Response Mapper

In [111]:
with open(MAPPER_PATH, "r", encoding="utf-8") as f:
    MAPPER = json.load(f)

### Testing Dataset

In [112]:
with open(TESTER_PATH, "r", encoding="utf-8") as f:
    TESTER = json.load(f)

### Embedder Model

In [113]:
embedder = SentenceTransformer(model_name)

## CaneSpeak

In [114]:
def cane_speak(question_text, disease_name="Healthy", score_threshold=0.50):
    qtext = question_text.strip()

    try:
        lang = detect(qtext)
    except LangDetectException:
        lang = "en"

    candidates = []
    for entry in MAPPER:
        disease = entry.get("disease", "").lower()
        if disease == disease_name.lower() or disease == "healthy":
            for qa in entry.get("quesans", []):
                subs = qa.get("sub_questions", {})
                
                if isinstance(subs, dict):
                    for lang_key, sub_list in subs.items():
                        for sq in sub_list:
                            candidates.append({
                                "canonical": qa.get("canonical_question", ""),
                                "sub_question": sq,
                                "answer": qa.get("answer", {}),
                                "lang_key": lang_key,
                            })
                else:
                    for sq in subs:
                        candidates.append({
                            "canonical": qa.get("canonical_question", ""),
                            "sub_question": sq,
                            "answer": qa.get("answer", {}),
                            "lang_key": "en",
                        })

    if not candidates:
        return {"success": False, "error": "No candidates found."}

    corpus = []
    for c in candidates:
        corpus.append(c["canonical"])
        corpus.append(c["sub_question"])

    corpus_emb = embedder.encode(corpus, convert_to_numpy=True)
    d = corpus_emb.shape[1]
    index = faiss.IndexFlatIP(d)
    faiss.normalize_L2(corpus_emb)
    index.add(corpus_emb)

    q_emb = embedder.encode([qtext], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    scores, idx = index.search(q_emb, k=1)
    best_vec_index = int(idx[0][0])
    best_candidate_index = best_vec_index // 2
    best_score = float(scores[0][0])

    result = candidates[best_candidate_index]

    if best_score < score_threshold:
        return {
            "success": False,
            "detected_language": lang,
            "score": best_score,
            "message": f"No sufficiently confident match. (Score={best_score:.2f})"
        }

    answer_field = result["answer"]
    if isinstance(answer_field, str):
        answer_text = answer_field
    else:
        answer_text = answer_field.get("en") or list(answer_field.values())[0]

    return {
        "success": True,
        "detected_language": lang,
        "score": best_score,
        "canonical_question": result["canonical"],
        "matched_sub_question": result["sub_question"],
        "answer_text": answer_text,
        "disease_name_used": disease_name
    }

## Test Function

In [115]:
def test_canespeak(tests = TESTER):

    correct = 0
    total = len(tests)

    y_true = []
    y_pred = []

    testing = []

    i = 0

    print("Testing Started.\n")
    for item in tests:
        q = item["question"]
        expected_disease = item["disease"].lower()

        res = cane_speak(q, disease_name=item["disease"], score_threshold=score_threshold)
        # print("\nQuestion:\t", q)

        if res["success"]:
            # print("Matched:\t", res["canonical_question"])
            # print("Answer\t:", res["answer_text"])
            # print("Score\t:", res["score"])

            canonical = res["canonical_question"].lower()
            answer = res["answer_text"].lower()

            disease_key = expected_disease[:3]

            match = disease_key in canonical or disease_key in answer

            y_true.append(1)
            y_pred.append(1 if match else 0)

            if match:
                correct += 1
                result = "Result:\tCorrect."

                testing.append({
                    "Question": q,
                    "Matched": res["canonical_question"],
                    "Answer": res["answer_text"],
                    "Score": res["score"],
                    "Result": result
                })
            else:
                result = "Result:\tIncorrect mapping."
                testing.append({
                    "Question": q,
                    "Matched": res["canonical_question"],
                    "Answer": res["answer_text"],
                    "Score": res["score"],
                    "Result": result
                })
        else:
            pass
            result = "Result:\tNo confident match. Score:", res.get("score")
            testing.append({
                "Question": q,
                "Matched": None,
                "Answer": None,
                "Score": None,
                "Result": result
            })
            y_true.append(1)
            y_pred.append(0)

        i += 1
        percentage = (i/total)
        if (percentage*100) % 10 == 0 and percentage != 0:
            print(f"{percentage:.0%}% complete.")
        

    print("\nTesting Ended.\n")

    accuracy = correct / total if total > 0 else 0
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    metrics = f"Accuracy:\t{accuracy:.2%}\nPrecision:\t{precision:.2%}\nRecall:\t\t{recall:.2%}\nF1 Score:\t{f1:.2%}"
    print(metrics)
    
    return testing, metrics

## Testing

In [116]:
testing, metrics = test_canespeak()

Testing Started.

10%% complete.
20%% complete.
30%% complete.
40%% complete.
50%% complete.
60%% complete.
70%% complete.
80%% complete.
90%% complete.
100%% complete.

Testing Ended.

Accuracy:	87.78%
Precision:	100.00%
Recall:		87.78%
F1 Score:	93.49%


## Saving Results

In [117]:
os.makedirs(SAVE_PATH, exist_ok=True)

In [118]:
with open(save_path_testing, "w", encoding="utf-8") as f:
    json.dump(testing, f, indent=4, ensure_ascii=False)
print(f"Saved testing results to: {save_path_testing}")

Saved testing results to: E:\Studies\MIT\7\FYP\Project\Results\CaneSpeak\all-MiniLM-L6-v2\2025-11-18_19-57\testing.json


In [119]:
with open(save_path_metrics, "w", encoding="utf-8") as f:
    f.write(metrics)
print(f"Saved metrics to: {save_path_metrics}")

Saved metrics to: E:\Studies\MIT\7\FYP\Project\Results\CaneSpeak\all-MiniLM-L6-v2\2025-11-18_19-57\metrics.txt
